In [ ]:
import geopandas as gpd
import pandas as pd
from pathlib import Path
from shapely.geometry import Polygon, MultiPolygon, GeometryCollection
from shapely.ops import unary_union
from shapely.validation import make_valid


## Script Overview:

# Main Preprocessing script that builds the data for the scrollys
# Additionally, I used QGIS to viualize the map datasets and do some manual inspection and checks to understand the data
# It takes the main dataset on historic territory and uses their year column to create snapshots of the developing territory over time
# Main input files are the indian concessions dataset and the historical US territory dataset
# It creates a full footprint of all native cessions and at each step calculates how much is overlapped by the US, adds the US territory to a counter and calculates remaining native area at that step.
# The final reservation layer is simply built with the "reservation current" categroy of the dataset
# Central functions are built to clean both files and adjust them to the same main histrorical timeline
# finally, output all results as jsons and geojsons, so they can be easily plugged into the html




# Adjust path files if necessary
INDIAN_FILE = Path("indian_concessions.json")
US_FILE = Path("US_HistStateTerr_Gen05.shp")
OUT_DIR = Path("data/data_outputs")

# Projected CRS for area calculations and simplification, set default full area
AREA_CRS = 5070
SQM_PER_SQMI = 2_589_988.110336

OUT_DIR.mkdir(exist_ok=True)


# These are the main scrolly steps
# Mostly, I used the simple Wikipedia article referenced in the html
# I cut down all of the steps into the most important ones for the map and territory displacements
timeline_steps = [
    [1, 1783, "1783-09-03", "Treaty of Paris"],
    [2, 1787, "1787-07-13", "Northwest Territory"],
    [3, 1803, "1803-12-20", "Louisiana Purchase"],
    [4, 1814, "1814-08-09", "Treaty of Fort Jackson / post-War of 1812 treaty wave"],
    [5, 1824, "1824-12-31", "Pre-removal treaty wave"],
    [6, 1830, "1830-05-28", "Indian Removal Act"],
    [7, 1838, "1838-12-31", "Trail of Tears"],
    [8, 1848, "1848-07-04", "Mexican Cession"],
    [9, 1851, "1851-12-31", "Reservation policy era"],
    [10, 1887, "1887-02-08", "Dawes Act / allotment era"],
    [11, 2010, "2000-12-31", "Present-day reservations"],
]

timeline = pd.DataFrame(
    timeline_steps,
    columns=["step", "year", "date", "title"]
)

timeline["date"] = pd.to_datetime(timeline["date"])


def require_columns(gdf, columns, file_name):
    missing = [col for col in columns if col not in gdf.columns]
    if missing:
        raise ValueError(f"{file_name} is missing columns: {missing}")


def polygonal_only(geom):
    # Keep only polygon geometry, since the map layers are polygon layers
    if geom is None or geom.is_empty:
        return None

    geom = make_valid(geom)

    if isinstance(geom, (Polygon, MultiPolygon)):
        return geom

    if isinstance(geom, GeometryCollection):
        polygons = [
            part for part in geom.geoms
            if isinstance(part, (Polygon, MultiPolygon)) and not part.is_empty
        ]
        return unary_union(polygons) if polygons else None

    return None


def clean_geometries(gdf):
    # Fix invalid geometries and remove anything that is not polygonal
    gdf = gdf.copy()
    gdf["geometry"] = gdf.geometry.apply(polygonal_only)
    gdf = gdf[gdf.geometry.notna() & ~gdf.geometry.is_empty].copy()
    return gdf


def simplify_for_web(gdf, tolerance_m):
    # Simplify in meters, then return to WGS84 for MapLibre
    gdf = clean_geometries(gdf).to_crs(AREA_CRS)
    gdf["geometry"] = gdf.geometry.simplify(
        tolerance=tolerance_m,
        preserve_topology=True
    )
    return clean_geometries(gdf).to_crs(4326)


def area_sqmi(gdf):
    # Calculate area in square miles
    if gdf.empty:
        return 0

    geom = unary_union(clean_geometries(gdf).to_crs(AREA_CRS).geometry)
    return round(geom.area / SQM_PER_SQMI)


# Load and clean Indigenous land layer
indian = gpd.read_file(INDIAN_FILE).to_crs(4326)

require_columns(
    indian,
    ["Cession_Reservation", "Year", "geometry"],
    INDIAN_FILE.name
)

indian["year"] = pd.to_numeric(indian["Year"], errors="coerce").astype("Int64")
indian = clean_geometries(indian)

cessions = indian[indian["Cession_Reservation"] == "Cession"].copy()
current_reservations = indian[indian["Cession_Reservation"] == "Reservation Current"].copy()


# Load and clean U.S. historical territory layer
us = gpd.read_file(US_FILE).to_crs(4326)

require_columns(
    us,
    ["START_DATE", "END_DATE", "geometry"],
    US_FILE.name
)

us["START_DATE"] = pd.to_datetime(us["START_DATE"], errors="coerce")
us["END_DATE"] = pd.to_datetime(us["END_DATE"], errors="coerce")
us = clean_geometries(us)


# Create one dissolved U.S. territory geometry for each scrolly step
us_rows = []

for _, step in timeline.iterrows():
    active = us[
        (us["START_DATE"] <= step["date"]) &
        (us["END_DATE"] >= step["date"])
    ].copy()

    if active.empty:
        continue

    active_proj = clean_geometries(active).to_crs(AREA_CRS)
    dissolved_geom = polygonal_only(unary_union(active_proj.geometry))

    if dissolved_geom is None:
        continue

    us_rows.append({
        "step": int(step["step"]),
        "scrolly_year": int(step["year"]),
        "selected_date": step["date"].strftime("%Y-%m-%d"),
        "event_title": step["title"],
        "area_sqmi_total": round(dissolved_geom.area / SQM_PER_SQMI),
        "geometry": dissolved_geom
    })

us_by_step = gpd.GeoDataFrame(us_rows, geometry="geometry", crs=AREA_CRS)
us_by_step_web = simplify_for_web(us_by_step, tolerance_m=5000)

us_by_step_web.to_file(
    OUT_DIR / "us_territory_snapshots_by_step_simplified.geojson",
    driver="GeoJSON"
)


# Create one full footprint of all recorded cessions
cessions_proj = clean_geometries(cessions).to_crs(AREA_CRS)
full_cession_geom = polygonal_only(unary_union(cessions_proj.geometry))

if full_cession_geom is None:
    raise ValueError("No usable cession geometry found.")

full_cession_area = round(full_cession_geom.area / SQM_PER_SQMI)

full_footprint = gpd.GeoDataFrame(
    [{
        "layer_type": "full_cession_footprint",
        "area_sqmi": full_cession_area,
        "geometry": full_cession_geom
    }],
    geometry="geometry",
    crs=AREA_CRS
)

full_footprint_web = simplify_for_web(full_footprint, tolerance_m=5000)

full_footprint_web.to_file(
    OUT_DIR / "ioa_full_cession_footprint.geojson",
    driver="GeoJSON"
)


# Create the current reservations layer for the final step
current_reservations = clean_geometries(current_reservations)

if not current_reservations.empty:
    current_reservations["step"] = 11
    current_reservations["scrolly_year"] = 2010
    current_reservations["event_title"] = "Present-day reservations"
    current_reservations["scrolly_type"] = "current_reservation"

    current_reservations_web = simplify_for_web(
        current_reservations,
        tolerance_m=3000
    )

    current_reservations_web.to_file(
        OUT_DIR / "ioa_current_reservations_final_step_simplified.geojson",
        driver="GeoJSON"
    )


# Create counters for how much of the full cession footprint overlaps U.S. territory
swallow_rows = []

for _, row in us_by_step.iterrows():
    us_geom = polygonal_only(row.geometry)

    if us_geom is None:
        continue

    swallowed_geom = polygonal_only(full_cession_geom.intersection(us_geom))
    remaining_geom = polygonal_only(full_cession_geom.difference(us_geom))

    swallowed_area = 0 if swallowed_geom is None else round(swallowed_geom.area / SQM_PER_SQMI)
    remaining_area = 0 if remaining_geom is None else round(remaining_geom.area / SQM_PER_SQMI)

    swallow_rows.append({
        "step": int(row["step"]),
        "year": int(row["scrolly_year"]),
        "event_title": row["event_title"],
        "full_cession_footprint_area_sqmi": full_cession_area,
        "swallowed_by_us_area_sqmi": swallowed_area,
        "remaining_outside_us_area_sqmi": remaining_area
    })

swallow_counters = pd.DataFrame(swallow_rows)

swallow_counters.to_json(
    OUT_DIR / "swallow_counters_by_step.json",
    orient="records",
    indent=2
)


# Create the main counter file used by the map
timeline_counters = timeline.merge(
    us_by_step.drop(columns="geometry"),
    on="step",
    how="left"
)

timeline_counters["current_reservation_features"] = 0
timeline_counters["current_reservation_area_sqmi"] = 0

if not current_reservations.empty:
    timeline_counters.loc[
        timeline_counters["step"] == 11,
        "current_reservation_features"
    ] = len(current_reservations)

    timeline_counters.loc[
        timeline_counters["step"] == 11,
        "current_reservation_area_sqmi"
    ] = area_sqmi(current_reservations)

timeline_counters["area_sqmi_total"] = (
    timeline_counters["area_sqmi_total"]
    .fillna(0)
    .round()
    .astype(int)
)

timeline_counters.to_json(
    OUT_DIR / "timeline_counters_clean.json",
    orient="records",
    indent=2,
    date_format="iso"
)

timeline.to_json(
    OUT_DIR / "timeline_steps.json",
    orient="records",
    indent=2,
    date_format="iso"
)

print("Preprocessing complete.")

Preprocessing complete.
